# 06 — Inventory of public 0.943 artifacts (CPU only)

This notebook inspects filenames and small JSON manifests from the public inputs used by the 0.943 inference notebook. It does not run any model, inspect hidden test data, or tune ensemble weights.

Expected settings: Accelerator **None**, Internet **Off**. Import this file as a new Kaggle notebook and verify that its Input sidebar includes the public datasets/notebooks/models from the original 0.943 notebook. Save & Run All, then download the output ZIP.


In [ ]:
from pathlib import Path
import csv, json, os, re

ROOT = Path('/kaggle/input')
OUT = Path('/kaggle/working')
if not ROOT.is_dir():
    raise FileNotFoundError('/kaggle/input is missing; run this notebook on Kaggle')
print('Mounted top-level inputs:', sorted(p.name for p in ROOT.iterdir()))
print('This is a CPU inventory; no GPU inference is performed.')

SKIP_DIRS = {'train_series', 'test_series', 'study_cache', 'dicom', 'dicoms', 'images', '__pycache__', '.git'}
KEYWORDS = re.compile(r'oof|out.of.fold|holdout|valid|gold|fold|report|label|manifest|metric|predict|preds?|split', re.I)
OOF_WORDS = re.compile(r'oof|out.of.fold|holdout|valid.*pred|pred.*valid', re.I)
MAX_FILES = 20000
seen = 0
stopped = False
rows = []
for base, dirs, files in os.walk(ROOT):
    base_path = Path(base)
    depth = len(base_path.relative_to(ROOT).parts)
    dirs[:] = sorted(d for d in dirs if d.lower() not in SKIP_DIRS and 'corpus' not in d.lower() and depth < 8)
    for name in files:
        seen += 1
        if seen > MAX_FILES:
            stopped = True; break
        path = base_path / name
        rel = str(path.relative_to(ROOT))
        if not KEYWORDS.search(rel) or path.suffix.lower() == '.dcm':
            continue
        try: size = path.stat().st_size
        except OSError: size = -1
        rows.append({'path': rel, 'bytes': size, 'suffix': path.suffix.lower(),
                     'likely_oof': bool(OOF_WORDS.search(rel))})
    if stopped: break

rows.sort(key=lambda x: (not x['likely_oof'], x['path']))
with (OUT / 'artifact_inventory.csv').open('w', newline='', encoding='utf-8') as fh:
    writer = csv.DictWriter(fh, fieldnames=['path', 'bytes', 'suffix', 'likely_oof'])
    writer.writeheader(); writer.writerows(rows)
print(f'Scanned {seen} filenames; saved {len(rows)} relevant entries; likely OOF files: {sum(r["likely_oof"] for r in rows)}')
if stopped: print('WARNING: filename scan hit its 20,000-file cap; inventory is partial')
for row in rows[:80]:
    print(('OOF? ' if row['likely_oof'] else '     ') + row['path'] + f' ({row["bytes"]} bytes)')


In [ ]:
# Inspect small JSON manifests only; never load model weights or image archives.
HINT = re.compile(r'oof|holdout|valid|gold|fold|train|label|selection|score|auc|split', re.I)
evidence = []
def walk_json(node, prefix='', limit=100):
    if limit <= 0: return []
    found = []
    if isinstance(node, dict):
        for key, value in node.items():
            here = f'{prefix}.{key}' if prefix else str(key)
            if HINT.search(str(key)) and not isinstance(value, (dict, list)):
                found.append((here, str(value)[:180]))
            if isinstance(value, (dict, list)):
                found.extend(walk_json(value, here, limit - len(found)))
            if len(found) >= limit: break
    elif isinstance(node, list):
        for i, value in enumerate(node[:30]):
            if isinstance(value, (dict, list)):
                found.extend(walk_json(value, f'{prefix}[{i}]', limit - len(found)))
            if len(found) >= limit: break
    return found[:limit]

for row in rows:
    if row['suffix'] != '.json' or not (0 <= row['bytes'] <= 2_000_000): continue
    path = ROOT / row['path']
    try:
        payload = json.loads(path.read_text(encoding='utf-8'))
        for key, value in walk_json(payload):
            evidence.append({'path': row['path'], 'key': key, 'value': value})
    except (OSError, UnicodeError, json.JSONDecodeError) as exc:
        evidence.append({'path': row['path'], 'key': '_read_error', 'value': str(exc)[:180]})
with (OUT / 'manifest_evidence.csv').open('w', newline='', encoding='utf-8') as fh:
    writer = csv.DictWriter(fh, fieldnames=['path', 'key', 'value'])
    writer.writeheader(); writer.writerows(evidence)
print(f'Saved {len(evidence)} manifest clues to manifest_evidence.csv')
print('If no trustworthy OOF prediction files are present, the next model must have its own grouped validation.')
